Loading Libraries

In [1]:
import pandas as pd
import numpy as np
import wbgapi as wb
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from statsmodels.stats.mediation import Mediation

Data Extraction and Consolidation

In [2]:
df_qarot = pd.read_csv('qarot_data.csv')[['code', 'year', 'implemented.quota']]
df_qarot.rename(columns={'code': 'country_code', 'implemented.quota': 'quota_active'}, inplace=True)

indicators = {'SG.GEN.PARL.ZS': 'women_in_parliament_pct', 
              'SL.TLF.CACT.FE.ZS': 'female_labor_participation_pct',
              'NY.GDP.PCAP.KD': 'gdp_per_capita'}

df_wb = wb.data.DataFrame(indicators.keys(), time=range(1995, 2016), columns='series').reset_index()
df_wb.rename(columns={**indicators, 'economy': 'country_code', 'time': 'year'}, inplace=True)
df_wb['year'] = df_wb['year'].str.replace('YR', '').astype(int)

df_final = pd.merge(df_wb, df_qarot, on=['country_code', 'year'], how='left')
df_final['quota_active'] = df_final['quota_active'].fillna(0)
df_final['log_gdp'] = np.log(df_final['gdp_per_capita'].replace(0, np.nan))

Data Cleaning and Preprocessing

In [3]:
df_clean = df_final.dropna(subset=['female_labor_participation_pct', 'women_in_parliament_pct', 'log_gdp']).copy()

def check_transition(group):
    has_treated_period = 1 in group['quota_active'].values
    has_control_period = 0 in group['quota_active'].values
    # Keep control-only countries AND treated countries with a pre-period baseline
    return (not has_treated_period) or (has_treated_period and has_control_period)

df_model = df_clean.groupby('country_code').filter(check_transition).copy()

if len(df_model) == 0:
    unmatched = set(df_qarot['country_code']) - set(df_wb['country_code'])
    raise ValueError(f"Merge Error. Unmatched codes: {list(unmatched)[:5]}")

df_model['country_id'], _ = pd.factorize(df_model['country_code'])
df_panel = df_model.set_index(['country_code', 'year'])

Model 1: DiD with Clustered SE

In [4]:
model_did = PanelOLS.from_formula(
    "female_labor_participation_pct ~ quota_active + log_gdp + EntityEffects + TimeEffects", 
    data=df_panel
)
res_did = model_did.fit(cov_type='clustered', cluster_entity=True)

Model 2: IV 

In [5]:
def within_demean(df, cols):
    out = df[cols].copy().astype(float)
    out -= out.groupby(level=0).transform('mean') # Entity
    out -= out.groupby(level=1).transform('mean') # Time
    out += out.mean() # Grand mean
    return out

dem_cols = ['female_labor_participation_pct', 'women_in_parliament_pct', 'quota_active', 'log_gdp']
df_dem = within_demean(df_panel, dem_cols)

model_iv = IV2SLS(
    dependent=df_dem['female_labor_participation_pct'],
    exog=df_dem[['log_gdp']],
    endog=df_dem[['women_in_parliament_pct']],
    instruments=df_dem[['quota_active']]
)
res_iv = model_iv.fit(cov_type='robust')

Model 3: Mediation

In [6]:
med_model = smf.ols("women_in_parliament_pct ~ quota_active + log_gdp + C(country_id) + C(year)", data=df_model)
out_model = smf.ols("female_labor_participation_pct ~ quota_active + women_in_parliament_pct + log_gdp + C(country_id) + C(year)", data=df_model)
med_engine = Mediation(out_model, med_model, exposure="quota_active", mediator="women_in_parliament_pct")
np.random.seed(42)
res_mediation = med_engine.fit(n_rep=500)

Results

In [8]:
print("DiD (Clustered SEs)")
print(f"  Coef : {res_did.params['quota_active']:+.4f}")
print(f"  SE   : {res_did.std_errors['quota_active']:.4f}")
print(f"  p    : {res_did.pvalues['quota_active']:.4f}")

DiD (Clustered SEs)
  Coef : +0.1841
  SE   : 0.4980
  p    : 0.7117


In [12]:
print("IV")
print(f"  LATE : {res_iv.params['women_in_parliament_pct']:+.4f}")
print(f"  SE   : {res_iv.std_errors['women_in_parliament_pct']:.4f}")
print(f"  p    : {res_iv.pvalues['women_in_parliament_pct']:.4f}")

IV
  LATE : +0.0332
  SE   : 0.0307
  p    : 0.2807


In [14]:
f_stat = res_iv.first_stage.diagnostics['f.stat'].iloc[0]
print(f"First-stage F : {f_stat:.2f} ({'strong' if f_stat > 10 else 'WEAK - caution'})")

print("Summary")
print(res_mediation.summary())

First-stage F : 251.40 (strong)
Summary
                          Estimate  Lower CI bound  Upper CI bound  P-value
ACME (control)            0.126941       -0.025203        0.259469    0.088
ACME (treated)            0.126941       -0.025203        0.259469    0.088
ADE (control)             0.066922       -0.334532        0.433406    0.716
ADE (treated)             0.066922       -0.334532        0.433406    0.716
Total effect              0.193863       -0.182940        0.550691    0.268
Prop. mediated (control)  0.463365       -6.829298        8.342890    0.324
Prop. mediated (treated)  0.463365       -6.829298        8.342890    0.324
ACME (average)            0.126941       -0.025203        0.259469    0.088
ADE (average)             0.066922       -0.334532        0.433406    0.716
Prop. mediated (average)  0.463365       -6.829298        8.342890    0.324


Checking for Parallel Trends

In [17]:
# Get first quota year per treated country
first_quota = (df_model[df_model['quota_active'] == 1]
               .groupby('country_code')['year']
               .min()
               .rename('first_quota_year'))

df_pt = df_model.merge(first_quota, on='country_code', how='left')

# Mark treated countries (those that ever adopt)
df_pt['ever_treated'] = df_pt['first_quota_year'].notna().astype(int)

# Keep only pre-quota rows for treated, and all rows for control
df_pt_pre = df_pt[
    (df_pt['ever_treated'] == 0) |  # all control rows
    (df_pt['year'] < df_pt['first_quota_year'])  # pre-period only for treated
].copy()

df_pt_pre['treated_x_year'] = df_pt_pre['ever_treated'] * df_pt_pre['year']

pt_test = smf.ols(
    "female_labor_participation_pct ~ treated_x_year + ever_treated + log_gdp + C(year)",
    data=df_pt_pre
).fit(cov_type='cluster', cov_kwds={'groups': df_pt_pre['country_code']})

print(f"Pre-period rows          : {len(df_pt_pre)}")
print(f"Ever-treated countries   : {df_pt_pre[df_pt_pre['ever_treated']==1]['country_code'].nunique()}")
print(f"Parallel trends coef     : {pt_test.params['treated_x_year']:.4f}")
print(f"Parallel trends p-value  : {pt_test.pvalues['treated_x_year']:.4f}")

Pre-period rows          : 3215
Ever-treated countries   : 57
Parallel trends coef     : 0.0925
Parallel trends p-value  : 0.7244
